# Setup Instructions (Dedicated Environment Recommended)
Due to version conflicts between PaddleOCR and WhisperX, it is highly recommended to run this notebook in a dedicated Python 3.11 environment.

**Run these commands in your terminal if you haven't already:**
```bash
conda create -n learning_chinese_as_a_journey_asr python=3.11 -y
conda activate learning_chinese_as_a_journey_asr
pip install numpy==2.0.2
pip install git+https://github.com/m-bain/whisperX.git
pip install pypinyin omegaconf torch tqdm
```


In [ ]:
import subprocess
import os
import re
from tqdm.notebook import tqdm

def get_duration(input_file):
    """Get the duration of the input file in seconds using ffprobe."""
    command = [
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', input_file
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None

def convert_m4a_to_mp4_with_progress(input_file, output_file):
    """
    Converts m4a to mp4 with a black screen and displays progress.
    """
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    duration = get_duration(input_file)
    if duration is None:
        print("Could not determine file duration. Progress bar might not be accurate.")
        duration = 1  # Fallback

    command = [
        'ffmpeg',
        '-f', 'lavfi',
        '-i', 'color=c=black:s=1280x720',
        '-i', input_file,
        '-c:v', 'libx264',
        '-preset', 'ultrafast',
        '-tune', 'stillimage',
        '-c:a', 'copy',
        '-pix_fmt', 'yuv420p',
        '-shortest',
        output_file,
        '-y',
        '-stats'
    ]

    print(f"Converting: {os.path.basename(input_file)}")
    
    # Use tqdm for progress bar
    pbar = tqdm(total=100, desc="Conversion Progress", unit="%")
    
    # Run ffmpeg and parse its stderr for progress
    process = subprocess.Popen(command, stderr=subprocess.PIPE, universal_newlines=True)
    
    # Pattern to match time=HH:MM:SS.ms in ffmpeg output
    time_pattern = re.compile(r"time=(\d+):(\d+):(\d+\.\d+)")
    
    last_pct = 0
    for line in process.stderr:
        match = time_pattern.search(line)
        if match:
            hours, minutes, seconds = map(float, match.groups())
            current_time = hours * 3600 + minutes * 60 + seconds
            pct = min(99, (current_time / duration) * 100)
            pbar.update(pct - last_pct)
            last_pct = pct
            
    process.wait()
    if process.returncode == 0:
        pbar.update(100 - last_pct)
        pbar.close()
        print(f"Successfully created: {output_file}")
    else:
        pbar.close()
        print(f"Error occurred during conversion. Exit code: {process.returncode}")

input_path = "/Users/hoangducanh/Documents/tmp_it/python/test_projects/learning_chinese_as_a_journey/notebook/data/7像素卷积核与CNN底层算法.m4a"
output_path = "/Users/hoangducanh/Documents/tmp_it/python/test_projects/learning_chinese_as_a_journey/notebook/data/7像素卷积核与CNN底层算法.mp4"

convert_m4a_to_mp4_with_progress(input_path, output_path)

In [ ]:
import torch
import omegaconf
import whisperx
import gc
from tqdm import tqdm
from pypinyin import pinyin, Style

# --- CONFIGURATION (OPTIMIZED FOR APPLE SILICON M1/M2/M3) ---
device = "cpu" 
compute_type = "int8" 
batch_size = 4
audio_file = "/Users/hoangducanh/Documents/tmp_it/python/test_projects/learning_chinese_as_a_journey/notebook/data/7像素卷积核与CNN底层算法.mp4"

# --- FIX: Patch for PyTorch 2.6 UnpicklingError ---
# Required because pyannote (VAD) uses omegaconf which PyTorch 2.6+ treats as "unsafe" by default.
torch.serialization.add_safe_globals([
    omegaconf.listconfig.ListConfig,
    omegaconf.dictconfig.DictConfig,
    omegaconf.nodes.AnyNode,
])


In [ ]:
# --- Step 1: Speech Recognition (Whisper V3) ---
print("--- Step 1: Recognizing Speech (Whisper V3) ---")

with tqdm(total=100, desc="Transcribing Audio", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}') as pbar:
    # On M1 CPU, "large-v3" is slow. Consider "base" if testing.
    model = whisperx.load_model("base", device, compute_type=compute_type) # small / base / large-v3
    audio = whisperx.load_audio(audio_file)
    result = model.transcribe(audio, batch_size=batch_size)
    pbar.update(100)

# Clear VRAM/RAM
model = None
gc.collect()

# --- Step 2: Time Alignment ---
print("\n--- Step 2: Aligning Timestamps ---")

# Load alignment model (Wav2Vec2 for Chinese)
model_a, metadata = whisperx.load_align_model(language_code="zh", device=device)

with tqdm(total=len(result["segments"]), desc="Aligning Segments") as pbar:
    result = whisperx.align(
        result["segments"], 
        model_a, 
        metadata, 
        audio, 
        device, 
        return_char_alignments=False
    )
    pbar.update(len(result["segments"]))

def get_pinyin(text):
    res = pinyin(text, style=Style.TONE)
    return " ".join([item[0] for item in res])

# --- TEST RESULTS ---
print("\n--- TEST RESULTS (First 5 Segments) ---")
for segment in result["segments"][:5]:
    cn_text = segment['text']
    pinyin_text = get_pinyin(cn_text)
    print(f"[{segment['start']:.2f}s]: {cn_text}")
    print(f"PY: {pinyin_text}\n")


--- Step 1: Recognizing Speech (Whisper V3) ---


Transcribing Audio:   0%|          | 0/100/opt/anaconda3/envs/learning_chinese_as_a_journey_asr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/learning_chinese_as_a_journey_asr/lib/python3.11/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.8.0) is not compatible with
             this version of TorchCodec. Refer to the version com

2026-04-19 11:23:22 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-04-19 11:23:22 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../../../opt/anaconda3/envs/learning_chinese_as_a_journey_asr/lib/python3.11/site-packages/whisperx/assets/pytorch_model.bin`


2026-04-19 11:23:57 - whisperx.asr - INFO - Detected language: zh (1.00) in first 30s of audio


Transcribing Audio: 100%|██████████| 100/100



--- Step 2: Aligning Timestamps ---


/opt/anaconda3/envs/learning_chinese_as_a_journey_asr/lib/python3.11/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Aligning Segments: 100%|██████████| 19/19 [02:14<00:00,  7.08s/it]


--- TEST RESULTS (First 5 Segments) ---
[0.99s]: 如果把一個數學掃描移的尺寸也就是從11個項速僅僅說小到7個項速你覺得能發生什麼?其實在2013年就是這一個微小的超參數改動讓圖像識別的錯誤率瞬間降了差不多5%
PY: rú guǒ bǎ yī gè shù xué sǎo miáo yí de chǐ cùn yě jiù shì cóng 11 gè xiàng sù jǐn jǐn shuō xiǎo dào 7 gè xiàng sù nǐ jué dé néng fā shēng shén me ? qí shí zài 2013 nián jiù shì zhè yī gè wēi xiǎo de chāo cān shù gǎi dòng ràng tú xiàng shí bié de cuò wù lǜ shùn jiān jiàng le chà bu duō 5%

[20.88s]: 对 这可以说直接鞏固了现代AI的革命 那欢迎你加入今天的深度资料探索今天我们要为你准备一场纯粹的技术解析 没错 我们要扒开这份详实的大学深度学习课程资料但你看看 卷机神经网络也就是CNN 它的底层引擎到底是怎么运转的
PY: duì   zhè kě yǐ shuō zhí jiē gǒng gù le xiàn dài AI de gé mìng   nà huān yíng nǐ jiā rù jīn tiān de shēn dù zī liào tàn suǒ jīn tiān wǒ men yào wèi nǐ zhǔn bèi yī cháng chún cuì de jì shù jiě xī   méi cuò   wǒ men yào bā kāi zhè fèn xiáng shí de dà xué shēn dù xué xí kè chéng zī liào dàn nǐ kàn kàn   juǎn jī shén jīng wǎng luò yě jiù shì CNN  tā de dǐ céng yǐn qíng dào dǐ shì zěn me yùn zhuàn de

[43.10s]: 我们今天不搞那种服鱼表面的客户,既然你热爱硬核知识,我们就直接聊算法公事,聊张亮数据聊

In [8]:
import os

def format_time_simple(seconds):
    """
    Converts total seconds into MM:SS format.
    Example: 789.0 -> 13:09
    """
    total_seconds = int(seconds)
    minutes, seconds = divmod(total_seconds, 60)
    # If your audio is longer than 60 mins, this still works (e.g., 65:01)
    return f"{minutes:02}:{seconds:02}"

# Updated path with .txt extension
output_path = "/Users/hoangducanh/Documents/tmp_it/python/test_projects/learning_chinese_as_a_journey/notebook/data/7像素卷积核与CNN底层算法.txt"

# Ensure the directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

print(f"--- Exporting transcript to: {output_path} ---")

try:
    with open(output_path, "w", encoding="utf-8") as f:
        for segment in result["segments"]:
            # Get start time in MM:SS
            start_time = format_time_simple(segment.get('start', 0.0))
            
            # Get Chinese text
            chinese_text = segment.get('text', '').strip()
            
            # Write to file in requested format
            f.write(f"{start_time}\n")
            f.write(f"{chinese_text}\n")
            
            # Optional: If you still want Pinyin included, uncomment the line below:
            # f.write(f"{get_pinyin(chinese_text)}\n")
            
    print("--- Export completed successfully! ---")
except Exception as e:
    print(f"--- Error writing to file: {e} ---")

--- Exporting transcript to: /Users/hoangducanh/Documents/tmp_it/python/test_projects/learning_chinese_as_a_journey/notebook/data/7像素卷积核与CNN底层算法.txt ---
--- Export completed successfully! ---


In [13]:
import os
from pathlib import Path
path = Path("..")
print(path)

..
